### **Notebook 0: Adquisición de datos**
##### **Objetivo:** Descargar los datos automáticamente de Google Drive o descomprimirlos a través de un archivo .zip local dentro de la carpeta de `Datos`.

##### **Procedimiento:** Este notebook realiza la descarga de los datos fuente GRD mediante:
1. **Descarga automática desde Google Drive**: Si la carpeta `Datos originales` no existe, se descargan los archivos `.txt` directamente desde Drive usando la librería `gdown`, que maneja correctamente los archivos grandes.
2. **Fallback al ZIP local**: Si la descarga falla, se asume que existe un archivo `Datos originales.zip` en la ruta `../../Datos` y se descomprime allí.
3. **Eliminación de redundancias**: Si aparece una carpeta duplicada (`Datos originales/Datos originales`), se mueven sus archivos a la carpeta principal y se elimina.
4. **Resultado final**: El usuario obtiene los archivos `.txt` listos en una sola carpeta, sin necesidad de descomprimir manualmente ni preocuparse por duplicados.

##### **Fuente original de datos:** Los archivos .txt del GRD público, fueron extraídos de datos abiertos de FONASA: https://public.tableau.com/views/PropuestaTableroGRD/PropuestaTableroGRD?%3AshowVizHome=no

##### **Fuente de respaldo de los datos (utilizadas para el proceso de adquisición):** https://drive.google.com/drive/folders/1zD3vm7UgkVBE50dETQlDX_b5yjS9wCFS?usp=drive_link

##### **Contenido de la carpeta de datos:** La carpeta de `Datos originales` debería tener 6 archivos `.txt` dentro:
1. *GRD_PUBLICO_2019.txt*
2. *GRD_PUBLICO_2020.txt*
3. *GRD_PUBLICO_2021.txt*
4. *GRD_PUBLICO_2022.txt*
5. *GRD_PUBLICO_2023.txt*
6. *GRD_PUBLICO_2024.txt*

**Nota:** Para la ejecución correcta de todos los notebooks, los datos `.txt` deben estar **obligatoriamente** en la siguiente ruta: `Branco-Garcia-Tesis-\Datos\Datos originales\...`



In [1]:
import os       # Gestión de rutas y verificación de existencia de archivos.
import shutil   # Movimiento de archivos entre directorios.
import zipfile  # Extracción de archivos comprimidos en formato ZIP.
import gdown    # Descarga de archivos alojados en Google Drive.

# Ruta donde deben quedar almacenados los archivos GRD originales.
ruta_padre = r"../../Datos/Datos originales"
# Ruta redundante que puede ser creada por una descompresión incorrecta.
ruta_redundante = os.path.join(ruta_padre, "Datos originales") # /Datos/Datos originales/Datos originales/
# Ruta alternativa de respaldo mediante archivo ZIP local.
ruta_zip = r"../../Datos/Datos originales.zip"
# Identificadores de Google Drive asociados a cada archivo GRD anual.
archivos_drive = {
    "GRD_PUBLICO_2019.txt": "1EeSoFmYrjF77tiRdCWguS3AioHHD4RfS",
    "GRD_PUBLICO_2020.txt": "1FohyVpNIPzxEA1MahuCFMxMd4sOfjHlf",
    "GRD_PUBLICO_2021.txt": "1XZO9dMWPZF2DlUl2LKnV87jcnsOUd4Jh",
    "GRD_PUBLICO_2022.txt": "1cT3F1QLyqdvKIibHpnJ2iVVjjXsjvEPz",
    "GRD_PUBLICO_2023.txt": "18nSVqalxGVuAe7JhVjs1qpX3KHMuMBfC",
    "GRD_PUBLICO_2024.txt": "1X7m0lNBzfx96XBu6td8CcHz6HcaqvTPV"
}
def descargar_de_drive(nombre, file_id, destino):
    """
    Descripción: Descarga un archivo GRD desde Google Drive y lo almacena en la ruta indicada.

    Entradas:
    - nombre: Nombre descriptivo del archivo (ej. "GRD_PUBLICO_2019.txt").
    - file_id: Identificador único del archivo en Google Drive.
    - destino: Ruta donde se guardará el archivo descargado.
    """
    # Construye la URL de descarga directa para Google Drive.
    url = f"https://drive.google.com/uc?id={file_id}"
    try:
        # Descarga el archivo en la ubicación especificada.
        gdown.download(url, destino, quiet=False)
        # Informa la descarga exitosa.
        print(f"Descargado: {nombre}")
    except Exception as e:
        # Reporta errores ocurridos durante la descarga.
        print(f"Error al descargar {nombre} desde Drive: {e}")

# Verifica si la carpeta principal de datos ya existe.
if not os.path.exists(ruta_padre):
    # Crea la estructura de directorios requerida.
    os.makedirs(ruta_padre, exist_ok=True)
    print(
        "Carpeta principal no encontrada. "
        "Intentando descargar archivos desde Google Drive..."
    )
    try:
        # Descarga todos los archivos GRD definidos en el diccionario.
        for nombre, file_id in archivos_drive.items():
            destino = os.path.join(ruta_padre, nombre) # Ruta completa para cada archivo descargado.
            descargar_de_drive(nombre, file_id, destino)

    except Exception as e: # En caso de cualquier excepción que ocurra durante el proceso de descarga.
        # Informa errores ocurridos durante el proceso de descarga.
        print(f"Error en descarga desde Drive: {e}")
        # Utiliza un archivo ZIP local como mecanismo alternativo.
        if os.path.exists(ruta_zip):
            print("Usando ZIP local como alternativa...")
            # Extrae el contenido del ZIP en la ubicación esperada.
            with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
                zip_ref.extractall(os.path.dirname(ruta_padre))
            print("Descompresión desde ZIP local completada.")
        else:
            # Reporta ausencia de fuentes de datos disponibles.
            print("ERROR: No existe carpeta ni ZIP disponible.")

# Si existe una carpeta redundante generada durante la extracción.
if os.path.exists(ruta_redundante): # En caso de que exista una carpeta /Datos originales/Datos originales.
    print("Carpeta redundante encontrada. Moviendo archivos...")
    # Obtiene el listado de archivos contenidos en la carpeta redundante.
    archivos = os.listdir(ruta_redundante)
    # Reubica cada archivo a la carpeta principal.
    for archivo in archivos: # Para cada archivo encontrado en la carpeta redundante.
        origen = os.path.join(ruta_redundante, archivo)
        destino = os.path.join(ruta_padre, archivo)
        # Mueve únicamente archivos que aún no existen en destino.
        if not os.path.exists(destino):
            shutil.move(origen, destino)
            print(f"Movido: {archivo}")
        else:
            # Evita sobrescribir archivos previamente existentes.
            print(f"Archivo ya existe, no se movió: {archivo}")
    # Elimina la carpeta redundante una vez vaciada.
    os.rmdir(ruta_redundante)
    print("\nÉXITO: Carpeta redundante eliminada con éxito.")

else: # En caso de que no se encuentre una carpeta redundante.
    # Informa que la estructura de carpetas ya es correcta.
    print("La carpeta redundante no existe o ya fue eliminada.")

Carpeta principal no encontrada. Intentando descargar archivos desde Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1EeSoFmYrjF77tiRdCWguS3AioHHD4RfS
From (redirected): https://drive.google.com/uc?id=1EeSoFmYrjF77tiRdCWguS3AioHHD4RfS&confirm=t&uuid=10b27d50-3c68-4485-9d7f-01c6d884c971
To: c:\Users\Carloto\Desktop\TESIS Código final\Branco-Garcia-Tesis-\Datos\Datos originales\GRD_PUBLICO_2019.txt
100%|██████████| 573M/573M [00:05<00:00, 100MB/s] 


Descargado: GRD_PUBLICO_2019.txt


Downloading...
From (original): https://drive.google.com/uc?id=1FohyVpNIPzxEA1MahuCFMxMd4sOfjHlf
From (redirected): https://drive.google.com/uc?id=1FohyVpNIPzxEA1MahuCFMxMd4sOfjHlf&confirm=t&uuid=68110325-0b4e-4b2f-94a4-e55b81cc5fd4
To: c:\Users\Carloto\Desktop\TESIS Código final\Branco-Garcia-Tesis-\Datos\Datos originales\GRD_PUBLICO_2020.txt
100%|██████████| 397M/397M [00:03<00:00, 109MB/s]  


Descargado: GRD_PUBLICO_2020.txt


...

Descargado: GRD_PUBLICO_2022.txt


Downloading...
From (original): https://drive.google.com/uc?id=18nSVqalxGVuAe7JhVjs1qpX3KHMuMBfC
From (redirected): https://drive.google.com/uc?id=18nSVqalxGVuAe7JhVjs1qpX3KHMuMBfC&confirm=t&uuid=52a0d7fe-2ba0-416d-a980-b8d0b81e70ca
To: c:\Users\Carloto\Desktop\TESIS Código final\Branco-Garcia-Tesis-\Datos\Datos originales\GRD_PUBLICO_2023.txt
100%|██████████| 1.08G/1.08G [00:18<00:00, 57.9MB/s]


Descargado: GRD_PUBLICO_2023.txt


Downloading...
From (original): https://drive.google.com/uc?id=1X7m0lNBzfx96XBu6td8CcHz6HcaqvTPV
From (redirected): https://drive.google.com/uc?id=1X7m0lNBzfx96XBu6td8CcHz6HcaqvTPV&confirm=t&uuid=0089db90-111e-4723-8d3f-d550aa547206
To: c:\Users\Carloto\Desktop\TESIS Código final\Branco-Garcia-Tesis-\Datos\Datos originales\GRD_PUBLICO_2024.txt
100%|██████████| 566M/566M [00:10<00:00, 52.2MB/s] 

Descargado: GRD_PUBLICO_2024.txt
La carpeta redundante no existe o ya fue eliminada.
